In [ ]:
from causallearn.utils.TXT2GeneralGraph import txt2generalgraph
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.Dataset import load_dataset
from causallearn.graph.ArrowConfusion import ArrowConfusion
from causallearn.graph.AdjacencyConfusion import AdjacencyConfusion
from causallearn.graph.SHD import SHD
from omegaconf import OmegaConf
from sklearn.metrics import roc_auc_score
from pathlib import Path
import numpy as np
import pandas as pd
from utils.plot import print_dict, plot_confusion
import mlflow
import matplotlib.pyplot as plt
import json

In [ ]:
# Log the experiment configuration
algorithm_tag = "pc_default"
data_tag = "csuite_cat_chain"
mlflow.end_run()
# Start MLflow run
mlflow.start_run(run_name=f"{algorithm_tag}/{data_tag}")

In [ ]:
# Load params
config = OmegaConf.load(f"config/{algorithm_tag}.yaml")
data_fpath = f"./data/csuite/{data_tag}/train.csv"
true_g_fpath = f"./data/csuite/{data_tag}/tetrad.txt"
mlflow.log_params({"algorithm": algorithm_tag, "dataset": data_tag, **config})

# Load data and run PC algorithm
true_g = txt2generalgraph(true_g_fpath)
data = pd.read_csv(data_fpath, header=None).to_numpy()

In [ ]:
cg = pc(
    data,
    alpha=config["alpha"],
    indep_test=config["indep_test"],
    stable=config["stable"],
    uc_rule=config["uc_rule"],
    uc_priority=config["uc_priority"],
    mvpc=config["mvpc"],
    correction_name=config["correction_name"],
    background_knowledge=config["background_knowledge"],
    verbose=False,
    show_progress=True,
)

In [ ]:
fig = plot_graphs_side_by_side(true_g, cg.G)

mlflow.log_figure(fig, "graphs_side_by_side.png")

# End MLflow run
mlflow.end_run()